## Setup

Runs on Colab, no-op locally. Run it first, before anything else.

`torch` and `numpy` are deliberately left alone: Colab's builds are CUDA-matched, and replacing them costs minutes and forces a runtime restart.

In [ ]:
# --- Colab setup ---------------------------------------------------
# Installs only what Colab is missing. Locally this whole cell is skipped.
import subprocess, sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers==5.7.0"],
        check=True,
    )
    print("Colab: dependencies installed.")
else:
    print("Local environment: nothing to install.")


# `notebooks/utils.py` is not on Colab's path, so `get_device` is defined here.
import torch


def get_device() -> torch.device:
    """Return the best available device for PyTorch operations."""
    if torch.cuda.is_available():
        print("Using GPU for PyTorch operations.")
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        print("Using Apple MPS for PyTorch operations.")
        return torch.device("mps")
    else:
        print("Using CPU for PyTorch operations.")
        return torch.device("cpu")


# A1 Introduction to BERT-style models

This notebook introduces a pretrained **encoder** model.

By the end you should be able to:

1. Turn text into the integers an LLM actually consumes, and back again.
2. Feed text into the model and collect its outputs.
3. Recover a hidden word from its context.
4. (Optional) Explain, from evidence you generated yourself, why this kind of model is **not** a
   text generator.

**Assumed:** comfortable Python. **Not assumed:** any PyTorch or Hugging Face.

## 0. The two libraries

We use two packages, `torch` and `transformers`. You do not need to know them well, but knowing
what each one is responsible for makes the code much easier to read.

### PyTorch (`torch`)

If you know `numpy`, you already know most of it: PyTorch's
`Tensor` is an n-dimensional array with essentially the same indexing, broadcasting and
reductions. It adds two things numpy lacks: it can run on a GPU, and it can track
gradients for training. The package also includes a lot of utilities and classes for
training and using neural networks.

The only PyTorch-specific functions in this notebook:

| Function | What it does |
|---|---|
| `torch.no_grad()` | Turns off gradient tracking. We are not training, so this saves memory and time. |
| `.to(device)` | Moves a tensor (or model) onto CPU / GPU / Apple MPS. |
| `.softmax(dim=-1)` | Turns raw scores into a probability distribution along the last axis. |
| `.topk(k)` | Returns the `k` largest values *and* their indices. |
| `.item()` | Pulls a single number out of a one-element tensor as a Python `float`. |

#### Helpful links
- [PyTorch documentation](https://pytorch.org/docs/stable/index.html)
- [Tensors in 10 minutes](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)
- [`torch.no_grad`](https://pytorch.org/docs/stable/generated/torch.no_grad.html)

### Hugging Face `transformers`

Hugging Face is a library of models: [official website](https://huggingface.co/).
It provides thousands of pretrained models, plus useful utilities, through one
interface, and it handles downloading and caching the weights for you.

Two objects matter:

- a **tokenizer**, which turns text into tokens, and
- a **model**, which maps those tokens into embeddings and processes them further.

The `Auto*` classes read the checkpoint's config and pick the right class for
you, so the same two lines work for **almost** any model on the Hub:

```python
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForMaskedLM.from_pretrained(model_id)
```

`AutoModelForMaskedLM` gives us a head that scores every vocabulary item at
every position, which is exactly what we need to fill in a `[MASK]`. GPT-style models use a
different class.

#### Helpful resources
- [`transformers` documentation](https://huggingface.co/docs/transformers/index)
- [Auto classes](https://huggingface.co/docs/transformers/model_doc/auto)
- [The Hugging Face NLP course](https://huggingface.co/learn/nlp-course)
- [Model Hub](https://huggingface.co/models)

### The model we use

[`answerdotai/ModernBERT-base`](https://huggingface.co/answerdotai/ModernBERT-base)
([paper](https://arxiv.org/abs/2412.13663), [announcement](https://huggingface.co/blog/modernbert))
is a 2024 redesign of the original
[BERT](https://arxiv.org/abs/1810.04805) model: same masked-language-modelling objective,
modernised architecture, much longer context.

The first run downloads roughly 600 MB and takes a minute or two.
Everything afterwards is served from a local cache. Two alternative checkpoints are listed in
the setup cell below.


### Setup

`get_device()` lives in `utils.py` and simply picks the best available backend
(CUDA, then Apple MPS, then CPU). On Colab's free tier this resolves to CPU, which is fine:
every cell in this notebook runs in well under a second once the model is loaded.

Two details in the cell below:

- `.eval()` puts the model in inference mode (it disables dropout). Always call it when
  you are not training; forgetting it makes results non-deterministic.
- `.to(device)` moves the weights. The model and its inputs must live on the same device.

We are working with `answerdotai/ModernBERT-base`; you can also try:

1. `answerdotai/ModernBERT-large`: same family, more parameters, larger download.
2. `google-bert/bert-base-uncased`: the 2018 original. Smaller, and a useful contrast, because
   it lowercases everything and tokenizes differently (see the note on `Ġ` below).


In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, BatchEncoding
import torch
# `get_device` is defined in the Setup cell at the top of this notebook.

model_id = "answerdotai/ModernBERT-base"
# model_id = "answerdotai/ModernBERT-large"  # alternative (LARGER) model
# model_id = "bert-base-uncased"  # alternative (SMALLER) model
device = get_device()

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(model_id).eval().to(device)

## 1. Tokenization

A model cannot read text. It reads **indices** into a fixed vocabulary that
was decided before pretraining began and can never change afterwards.

Tokenization is that translation step, and it is worth spending real time on: almost
every confusing result later in this notebook traces back to a misunderstanding here.

The vocabulary is just a dictionary mapping token strings to integer ids. ModernBERT's has
**50,368** tokens, far fewer than the number of English words, which is the whole point:
rare words get split into several pieces rather than being lost.

In [ ]:
vocab = tokenizer.get_vocab()
print(f"{len(vocab):,} entries\n")

# A sample, sorted by id so it is the same on every machine and every run.
by_id = sorted(vocab.items(), key=lambda kv: kv[1])
for token, i in by_id[:4] + by_id[9000:9004] + by_id[-4:]:
    print(f"  {i:>6}  {token!r}")

`tokenizer.tokenize()` shows the pieces as strings, without converting them to ids yet.
This is the single most useful debugging call in the library.

Two things to notice in the output:

**`Ġ` means "preceded by a space".** ModernBERT uses byte-pair encoding, which encodes
whitespace *into* the token. So `'Ġcity'` is the token for "␣city", and `'city'` (no `Ġ`)
is a different vocabulary entry used when the word starts a line or follows an opening
bracket. This trips people up constantly: if you look up a word's probability using the
version without the leading space, you will get a number that is wrong by orders of
magnitude. 

> Different models use different conventions. `bert-base-uncased` uses WordPiece, which
> marks *continuations* with `##` instead (`playing` → `play`, `##ing`) and has no space
> marker at all. Neither is more correct; they are just different encodings.

**Words are not tokens.** Common words are single tokens, rare ones are split. This is
why "how many tokens is my text?" never has an obvious answer.

In [ ]:
input_text = "The city of Oxford is located in [MASK]."
f"Tokenized text: {tokenizer.tokenize(input_text)}"

Calling the tokenizer directly (rather than `.tokenize()`) gives you the model-ready
inputs. `return_tensors="pt"` asks for PyTorch tensors rather than plain lists.

The result is a dict-like `BatchEncoding` containing:

- **`input_ids`**: the integer sequence. Shape `(batch, seq_len)`; the leading `1` is a
  batch of one sentence.
- **`attention_mask`**: 1 for real tokens, 0 for padding. Only meaningful when you batch
  sentences of different lengths, so we switch it off here for readability.

Notice the sequence is **two tokens longer** than the tokenized text above. The tokenizer
adds special tokens automatically:

| Token | Role |
|---|---|
| `[CLS]` | Start-of-sequence. Its final vector is conventionally used as a whole-sequence summary. |
| `[SEP]` | End-of-sequence separator. |
| `[MASK]` | The blank we are asking the model to fill. |

`[MASK]` is not a magic string. It is a genuine vocabulary entry (id `50284`) that the
model saw constantly during pretraining.

In [ ]:
tokenized = tokenizer(input_text, return_tensors="pt", return_attention_mask=False)
tokenized

`decode()` goes back the other way: ids → text. Useful as a sanity check that the model
sees what you think it sees.

`clean_up_tokenization_spaces=False` keeps the raw reconstruction. Note that the special
tokens are included and spacing looks slightly off, because decoding is lossy at the
margins. Treat it as a diagnostic rather than a faithful inverse.

In [ ]:
tokenizer.decode(tokenized["input_ids"][0], clean_up_tokenization_spaces=False)

### ✏️ Exercise 1: typos, and what "unknown word" really means

Every word so far has been clean and common. Run the cell on a mix of ordinary words,
misspellings and things no corpus contains, then answer three questions:

1. What happens to the **number of tokens** when a word is misspelled?
2. What happens if you use a non-Latin character?
3. Which of these produce the `[UNK]` token?

In [ ]:
WORDS = [
    "hospital",
    # add your own here
]

for word in WORDS:
    pieces = tokenizer.tokenize(" " + word)
    ids = tokenizer(" " + word, add_special_tokens=False)["input_ids"]
    flag = "[UNK]!" if tokenizer.unk_token_id in ids else ""
    print(f"{word:<32} {len(pieces):>2} tokens {flag:<7} {pieces}")

You can watch what that costs. `"The city of Oxford is located in [MASK]."` gives `' England'`
at 66.4%. Change one letter to get `Oxfrod` and `' England'` falls to 13.2%, losing first
place to `' Ireland'`. Adding more context does not repair it.

<details>
<summary><b>Answer</b> (open once you have run it)</summary>

My list:
```
    "hosptial",  # one letter transposed
    "Oxford",
    "Oxfrod",
    "tokenization",
    "antidisestablishmentarianism",
    "zyzzyva",
    "gpt4", "слово"
```

1. **Typos are expensive.** `hospital` is a single token; `hosptial` becomes four,
`['Ġh', 'os', 'pt', 'ial']`. `Oxford` is one token, `Oxfrod` is three.
2. **Uncommon characters** always have something to map to.
3. **`[UNK]` never appears.** Not for `zyzzyva`, not for emoji, not for runes. Byte-pair encoding
falls back to shorter and shorter pieces and, in the limit, to individual bytes, so *every*
string is representable. 

That is a property of BPE, not of tokenizers in general. WordPiece models such as
`bert-base-uncased` do emit `[UNK]`, which is the convention you will see on the tokenization
slide.

</details>

## 2. Running the model

Now the forward pass. Three lines: tokenize, run, read the output.

`model(**tokenized)` unpacks the `BatchEncoding` into keyword arguments
(`input_ids=..., attention_mask=...`). This is why the tokenizer and model pair so
conveniently: the tokenizer's output keys are exactly the model's argument names.

`torch.no_grad()` disables gradient bookkeeping. We are not training, so it is pure
overhead without it.

In [ ]:
input_text = "The city of Oxford is located in [MASK]."
tokenized = tokenizer(input_text, return_tensors="pt").to(device)
with torch.no_grad():  # disable gradient calculation for inference (saves memory)
    outputs = model(**tokenized)
logits = outputs.logits

### Reading the output

The model returns **logits**: raw, unnormalised scores (`softmax` converts scores to probabilities):
```
(batch, seq_len, vocab_size)  =  (1, 11, 50368) #shape
```

That is a score for **every vocabulary item at every position**. The model did not just
predict the mask; it produced a full 50,368-way distribution at all 11 positions, in one
pass.



In [ ]:
print(f"logits shape: {tuple(logits.shape)}  = (batch, seq_len, vocab_size)\n")

probs = logits[0].softmax(dim=-1)  # a distribution at EVERY position
top1 = probs.argmax(dim=-1)

print(f"{'pos':>3}  {'input':>12}  {'top-1 pred':>12}  {'p(top1)':>8}  {'p(input)':>8}")
for i, (tid, ptid) in enumerate(zip(tokenized["input_ids"][0].tolist(), top1.tolist())):
    print(
        f"{i:>3}  {tokenizer.decode([tid], clean_up_tokenization_spaces=False)!r:>12}  {tokenizer.decode([ptid], clean_up_tokenization_spaces=False)!r:>12}"
        f"  {probs[i, ptid]:>8.3f}  {probs[i, tid]:>8.3f}"
    )

### Look at the table before reading on

1. At almost every position `p(input)` is 1.000. Why is copying so easy for the model?
2. Position 8 behaves differently from every other row. What is different about it?
3. What happens to `p(England)` if you misspell some words?

<details>
<summary><b>Answer</b></summary>

1. **At every unmasked position the prediction is the input, at p ≈ 1.000.** These are real
predictions, just trivial ones: the model learned to copy.

2. **Position 8 is the only interesting row.** It is the only place with no answer to copy, and
the only row where confidence falls below 1.000 and `p(input)` collapses to 0.000.

3. Misspellings would reduce `p(England)`.

Why is copying so easy? Because during pretraining the loss was only ever computed at *masked*
positions. Everywhere else the label is set to `-100` and ignored. The model was never graded
on the visible positions, so it has no incentive to do anything but pass them through.

</details>

### Just the mask

In practice we only want the masked position, so we locate it and slice.

In [ ]:
# To get predictions for the mask:
masked_index = tokenized["input_ids"][0].tolist().index(tokenizer.mask_token_id)
predicted_token_id = outputs.logits[0, masked_index].argmax(axis=-1)
predicted_token = tokenizer.decode(predicted_token_id)
print("Predicted token:", predicted_token)
# Predicted token:  England

`topk(5)` gives the five highest-scoring tokens and their ids. This is the "fill in the
blank" behaviour BERT is known for.

Note the leading spaces in the results (`' England'`, not `'England'`). That is the `Ġ`
convention from section 1 showing up in the output.

In [ ]:
# where is the [MASK]?
mask_pos = (tokenized["input_ids"][0] == tokenizer.mask_token_id).nonzero(
    as_tuple=True
)[0]

with torch.no_grad():
    logits = model(**tokenized).logits

# logits at the mask position -> probabilities over the whole vocabulary
probs = logits[0, mask_pos[0]].softmax(dim=-1)
top = probs.topk(5)

print(f"Top 5 tokens for [MASK] in: {input_text!r}")
print(f"{'Score':>6}  {'Token':>12}  {'ID':>4}")
for score, tid in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{score * 100:>5.2f}%:  {tokenizer.decode([tid])!r:>12} ({tid})")

### ✏️ Exercise 2: does it check whether the sentence is true?

It seems that the model has no incentive to do anything but copy what it can see (on unmasked positions).
Test it. Put something false into the sentence and check whether the model objects.

In [ ]:
COUNTRY = "England"

text = f"The city of Oxford is located in {COUNTRY}. It is known for its [MASK] university."
enc = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    probs = model(**enc).logits[0].softmax(dim=-1)

top1 = probs.argmax(dim=-1)

print(
    f"{'pos':>3}  {'input':>14}  {'top-1 pred':>14}  {'p(top1)':>10}  {'p(input)':>10}"
)
for i, (tid, ptid) in enumerate(zip(enc["input_ids"][0].tolist(), top1.tolist())):
    print(
        f"{i:>3}  {tokenizer.decode([tid], clean_up_tokenization_spaces=False)!r:>14}  {tokenizer.decode([ptid], clean_up_tokenization_spaces=False)!r:>14}"
        f"  {probs[i, ptid]:>10.3f}  {probs[i, tid]:>10.3f}"
    )

<details>
<summary><b>Answer</b> (open once you have run it)</summary>

I used `COUNTRY=Peru`.

1. **It reduces the certainty.** `p('Peru')` at its own position is **0.626**, though that is not enough to correct the sentence.

2. The masked word predicts  `' Oxford'`, but at **19.3%** instead of the 59.2% it managed with `England`. The false country makes the rest of the sentence harder to reconstruct.


</details>

## 3. Bidirectional context

The "B" in BERT stands for **bidirectional**: every position attends to tokens on *both*
sides. This is the defining property of the encoder family, and it is easy to demonstrate.

We use two masks in one sentence, where the second sentence adds information relevant to
the first mask.

A small helper, since we look up mask positions repeatedly. `nonzero` returns the indices
where the condition holds, and there may be several.

In [ ]:
def mask_positions(tokenized_input: BatchEncoding) -> torch.Tensor:
    """Return all positions of the mask token in the tokenized input."""
    return (tokenized_input["input_ids"][0] == tokenizer.mask_token_id).nonzero(
        as_tuple=True
    )[0]


Re-tokenize and re-run the model for the new sentence.

> **A trap worth naming.** `logits` must be recomputed whenever `input_text` changes. If
> you edit the sentence but reuse stale `logits` from an earlier cell, you either get an
> `IndexError` (if the new sentence is longer) or, far worse, a plausible-looking number
> computed from the *previous* sentence. Binding `tokenized` and `logits` in the same cell
> is the reliable defence, and it is why they sit together below.

In [ ]:
input_text = (
    "The city of Oxford is located in [MASK]. It is known for its [MASK] university."
)

In [ ]:
tokenized = tokenizer(input_text, return_tensors="pt", return_attention_mask=False).to(
    device
)
mask_pos = mask_positions(tokenized)

with torch.no_grad():
    logits = model(**tokenized).logits

In [ ]:
# logits at the mask position -> probabilities over the whole vocabulary
for pos in mask_pos:
    probs = logits[0, pos].softmax(dim=-1)
    top = probs.topk(5)

    print(f"Top 5 tokens for [MASK] at position {pos} in: {input_text!r}")
    print(f"{'Score':>6}  {'Token':>12}  {'ID':>4}")
    for score, tid in zip(top.values.tolist(), top.indices.tolist()):
        print(f"{score * 100:>5.2f}%:  {tokenizer.decode([tid])!r:>12} ({tid})")
    print()  # Add a blank line between the two mask positions


### What just happened

Compare the first mask against section 2, where the sentence stopped after "located in
[MASK]":

| Context | p(England) |
|---|---|
| One sentence | 66.4% |
| Plus "known for its ___ university" | **94.3%** |

The mask position and everything to its left are unchanged. The only new information is to
its **right**, and the prediction sharpened by 28 points. That is bidirectional attention
doing work, and it is precisely what an autoregressive (left-to-right) model
cannot do.

### ✏️ Exercise 3: how many blanks can it fill at once?

Section 3 made two claims: that context flows both ways, and that all the masks are filled in
a **single forward pass without seeing each other**.

First, the mask budget. Run the cell and watch what happens as blanks are added to the same
sentence.

In [ ]:
SENTENCES = {
    1: "The city of Oxford is located in [MASK] and is famous for its university.",
    2: "The city of [MASK] is located in [MASK] and is famous for its university.",
    # add a sentence of your own
}

for n, text in SENTENCES.items():
    enc = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        lg = model(**enc).logits
    filled = []
    for position in mask_positions(enc):
        distribution = lg[0, position].softmax(dim=-1)
        filled.append(
            f"{tokenizer.decode([distribution.argmax()]).strip()!r} "
            f"{distribution.max():.0%}"
        )
    print(f"{n} mask(s): " + ", ".join(filled))

<details>
<summary><b>Code (Help)</b></summary>

```
SENTENCES = {
    1: "The city of Oxford is located in [MASK] and is famous for its university.",
    2: "The city of [MASK] is located in [MASK] and is famous for its university.",
    3: "The [MASK] of [MASK] is located in [MASK] and is famous for its university.",
    4: "The [MASK] of [MASK] is located in [MASK] and is [MASK] for its university.",
    5: "The [MASK] of [MASK] is [MASK] in [MASK] and is famous for its [MASK].",
    6: "The [MASK] of [MASK] is [MASK] in [MASK] and is [MASK] for its [MASK].",
}

```
</details>

1. Does the model get less confident as we add more masks?
2. Does it stay coherent (e.g., does the resulting sentence has a correct grammar)?

<details>
<summary><b>Answers</b></summary>

**The mask budget:** 
1. One blank: `'England'` at 95%.
2. Two blanks: the model drops to `'Paris'` at 4% and `'Germany'` at 8% 
3. By five blanks it is confidently wrong, `'located'` at 83% and `'University'` at 81%. 
4. **Confidence goes back up** as accuracy collapses: with little context left, the model falls back on generic
sentence shape, and generic is something it is very sure about.

Rule of thumb: masked language models are trained at roughly 15–30% masking. Push far past
that at inference and you are outside what the model ever practised.

</details>

#### Another example

In [ ]:
input_text = "[MASK] is the capital of [MASK]."


enc = tokenizer(input_text, return_tensors="pt").to(device)
with torch.no_grad():
    lg = model(**enc).logits
filled = []
for position in mask_positions(enc):
    distribution = lg[0, position].softmax(dim=-1)
    filled.append(
        f"{tokenizer.decode([distribution.argmax()]).strip()!r} "
        f"{distribution.max():.0%}"
    )
print("mask(s): " + ", ".join(filled))

<details>
<summary><b>Answer</b> (Open after you run the cell above)</summary>

**The contradiction.** `[MASK] is the capital of [MASK]` gives `'London'` (29.7%) for the
first and `' India'` (8.6%) for the second. The model's single most likely reading is *London
is the capital of India*.

That is not a bug, it is the architecture. Both blanks were scored in one forward pass, and
neither could see the other's tokens. Each is individually reasonable; jointly they are nonsense.

This is exactly why an encoder cannot generate coherent text.

</details>

## 4. What BERT tells us about the underlying data?

A masked language model gives you a full probability distribution over 50,368 tokens at any
blank you choose to put in a sentence. That distribution is a **measurement of the text the
model was trained on**.


In [ ]:
import pandas as pd


def mask_distribution(text: str) -> torch.Tensor:
    """Probability over the whole vocabulary at the [MASK] in `text`."""
    enc = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return logits[0, mask_positions(enc)[0]].softmax(dim=-1)


def probability_of(distribution: torch.Tensor, word: str) -> float | None:
    """p(word) at the mask. None if `word` is not a single token, see section 5."""
    ids = tokenizer(" " + word, add_special_tokens=False)["input_ids"]
    return distribution[ids[0]].item() if len(ids) == 1 else None


def top_k(distribution: torch.Tensor, k: int = 5) -> pd.DataFrame:
    """The `k` most probable tokens at the mask, as a table of p and token string."""
    top = distribution.topk(k)
    return pd.DataFrame(
        {
            "p": [round(v, 4) for v in top.values.tolist()],
            "token": [
                tokenizer.decode([i], clean_up_tokenization_spaces=False).strip()
                for i in top.indices.tolist()
            ],
        }
    )


#### Case study: Human-like biases

[Caliskan et al. (2017)](https://www.science.org/doi/10.1126/science.aal4230)
and [Garg et al. (2018)](https://www.pnas.org/doi/10.1073/pnas.1720347115) looked at the outputs of BERT-like models to see if they embed human-like biases.


A **contrast** between two distributions that differ in
one word is closer to an experiment: hold the frame fixed, vary the thing you care about, and
the difference is attributable to that thing.

In [ ]:
male = mask_distribution("The man worked as a [MASK].")
female = mask_distribution("The woman worked as a [MASK].")

print("The man worked as a ...")
print(top_k(male).to_string(index=False))
print("\nThe woman worked as a ...")
print(top_k(female).to_string(index=False))

print("\nratio p(word | woman) / p(word | man)")
for word in ["nurse", "secretary", "teacher", "lawyer", "farmer", "engineer"]:
    m, f = probability_of(male, word), probability_of(female, word)
    print(f"  {word:<12} {f / m:>6.1f}x     (man {m:.4f}, woman {f:.4f})")

`nurse` is **11.6 times** more likely after *woman* than after *man*; `farmer` and `engineer`
run the other way. Nothing here is a mistake by the model. It is reporting an association present in the
training corpus, namely how text tends to write about occupations.

#### Case study: Human-like biases II

In the actual study, [Caliskan et al. (2017)](https://www.science.org/doi/10.1126/science.aal4230)
and [Garg et al. (2018)](https://www.pnas.org/doi/10.1073/pnas.1720347115) swept over multiple professions and looked at how the probability of certain words changed.


In [ ]:
OCCUPATIONS = [
    "nurse",
    "receptionist",
    "teacher",
    "cleaner",
    "secretary",
    "scientist",
    "lawyer",
    "doctor",
    "engineer",
    "plumber",
]

rows = []
for job in OCCUPATIONS:
    d = mask_distribution(
        f"The {job} finished the shift and said [MASK] would call back later."
    )
    she, he = probability_of(d, "she"), probability_of(d, "he")
    rows.append(
        {
            "occupation": job,
            "p(she)": round(she, 3),
            "p(he)": round(he, 3),
            "she-he": round(she - he, 2),
        }
    )

pd.DataFrame(rows).sort_values("she-he", ascending=False)

**Question:** Does this effect survive if you rephrase the template?
Try: `The {job} handed in the report and joked that [MASK] deserved a raise.`
   

<details>
<summary><b>What to check before interpreting the results?</b></summary>

Four things, roughly in order of how often they matter.

**Does the result survive a reworded frame?** If the effect appears with *"worked as a"* and
vanishes with *"had a job as a"*, you measured the frame, not the association. Report a few
paraphrases, not your best one.

**Is there a baseline frame?** A ratio of 11x means little until you know what a neutral frame
gives. Run one where you expect no effect and see what the noise floor looks like.

**Which corpus is this?** ModernBERT was trained on web text to 2024. A finding here is about
that text. Garg et al. could talk about *change* over a century precisely because they trained
separate models on separate decades. One model gives you one snapshot.

</details>

## 5. Scoring multi-token options and synonym search

To compare options of different lengths, write each candidate into the sentence, mask its
tokens **one at a time**, and average the log-probabilities. This is called
**pseudo-log-likelihood** ([Salazar et al., 2020](https://arxiv.org/abs/1910.14659)).

Higher (less negative) is better.

In [ ]:
def score_option(template: str, option: str) -> float:
    """Pseudo-log-likelihood of `option` filling {} in `template`, per token.

    A simple version of a scoring function that works for options spanning several tokens.
    """
    prefix, suffix = template.split("{}")
    pre_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
    opt_ids = tokenizer(option, add_special_tokens=False)["input_ids"]
    suf_ids = tokenizer(suffix, add_special_tokens=False)["input_ids"]
    ids = torch.tensor(
        [[tokenizer.cls_token_id, *pre_ids, *opt_ids, *suf_ids, tokenizer.sep_token_id]]
    ).to(device)
    # print("Sequence of token IDs:", ids.tolist())
    # print("Decoded:", tokenizer.decode(ids[0], clean_up_tokenization_spaces=False))
    start = 1 + len(pre_ids)

    total = 0.0
    for k in range(len(opt_ids)):  # mask one option token at a time
        masked = ids.clone()
        masked[0, start + k] = tokenizer.mask_token_id
        with torch.no_grad():
            logits = model(input_ids=masked).logits
        total += logits[0, start + k].log_softmax(-1)[opt_ids[k]].item()
    return total / len(opt_ids)

In [ ]:
template = "The closest synonym to social is {}."
scores = {
    o: score_option(template, o)
    for o in [
        "programming",
        "cooking",
        "painting",
        "reading",
    ]
}

print(f"{'Option':>12}  {'Score (log-likelihood)':>22}")

for o, s in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"{o:>12}  {s:>20.4f}")

**Question:** Which words come out closest to `social`, and which come out furthest away?
<details>
<summary><b>My Results</b></summary>

The closest synonym that I have found is `singing`, and the furthest away is `running`.

</details>

## 6. What to take away

1. You have seen how tokenization works.
2. You have seen how masked-language modelling works, and how it differs from next-token
   generation.
3. You have seen what you can do with the outputs of these models.

### (Optional) If you want to learn more

- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/): the standard visual explanation
- [Hugging Face NLP course, ch. 1-2](https://huggingface.co/learn/nlp-course/chapter1/1)
- [BERT (Devlin et al., 2018)](https://arxiv.org/abs/1810.04805): the original masked-language-modelling paper
- [ModernBERT (Warner et al., 2024)](https://arxiv.org/abs/2412.13663): what changed in six years
- [Caliskan et al. (2017)](https://www.science.org/doi/10.1126/science.aal4230) and
  [Garg et al. (2018)](https://www.pnas.org/doi/10.1073/pnas.1720347115): corpus association
  measured at scale, and turned into social science
